# EV Model Visualization (matplotlib)

This notebook demonstrates how to load a saved model and dataset from the repository and visualize model predictions using matplotlib.

Plots included:
- Actual vs Predicted scatter with y=x reference
- Residuals vs Predicted values
- Residuals histogram
- Feature importances (if available on the model)

Adjust the `target` detection logic below if you want a specific target column.

In [3]:
# Imports
import os
import glob
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

plt.style.use('seaborn')

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# Diagnostic: show which Python executable and matplotlib file the kernel will use
import sys, importlib
print('sys.executable:', sys.executable)
print('sys.version:', sys.version.replace('\n', ' '))
import matplotlib
print('matplotlib.__file__:', getattr(matplotlib, '__file__', None))
print('matplotlib version:', matplotlib.__version__)

In [ ]:
# Load dataset: prefer scaled features if available, otherwise fallback to cleaned CSV
if os.path.exists('features_standard_scaled.csv'):
    df = pd.read_csv('features_standard_scaled.csv')
    print('Loaded features_standard_scaled.csv')
elif os.path.exists('features_minmax_scaled.csv'):
    df = pd.read_csv('features_minmax_scaled.csv')
    print('Loaded features_minmax_scaled.csv')
elif os.path.exists('electric_vehicles_spec_2025_cleaned.csv'):
    df = pd.read_csv('electric_vehicles_spec_2025_cleaned.csv')
    print('Loaded electric_vehicles_spec_2025_cleaned.csv')
else:
    raise FileNotFoundError('No suitable data file found in repo root.')

print('Data shape:', df.shape)
df.head()

In [ ]:
# Detect numeric columns and pick a target variable automatically (editable)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print('Numeric columns detected:', numeric_cols)
# Heuristic to find a price-like target
candidates = [c for c in numeric_cols if any(k in c.lower() for k in ('price','cost','msrp','amount'))]

In [ ]:
# Choose target column (auto-select or change manually below)
if len(candidates):
    target = candidates[0]
    print('Auto-selected target:', target)
else:
    # default: choose the last numeric column (please change if incorrect)
    if len(numeric_cols) == 0:
        raise ValueError('No numeric columns found to use as features/target.')
    target = numeric_cols[-1]
    print('Defaulting to last numeric column as target:', target)

# Prepare X and y (only numeric features)
X = df.drop(columns=[target])
X = X.select_dtypes(include=[np.number]).copy()
y = df[target].copy()
feature_names = X.columns.tolist()
print('Feature matrix shape:', X.shape)
print('Target vector shape:', y.shape)
X.head()

In [ ]:
# Load a saved model: prefer GradientBoosting if present, otherwise take the first model_*.joblib
preferred = 'model_GradientBoosting.joblib'
if os.path.exists(preferred):
    model_path = preferred
elif len(glob.glob('model_*.joblib'))>0:
    model_path = glob.glob('model_*.joblib')[0]
else:
    raise FileNotFoundError('No model_*.joblib file found in repository root.')

model = joblib.load(model_path)
print('Loaded model:', model_path)
model

In [ ]:
# Predict and evaluate
y_pred = model.predict(X)
mse = mean_squared_error(y, y_pred)
mae = mean_absolute_error(y, y_pred)
r2 = r2_score(y, y_pred)
print(f'MSE: {mse:.4f}, MAE: {mae:.4f}, R2: {r2:.4f}')

In [ ]:
# Plotting: Actual vs Predicted, Residuals, Residual histogram, Feature importances (if present)
residuals = y - y_pred
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
ax = axes[0,0]
ax.scatter(y, y_pred, alpha=0.6, edgecolor='k')
lims = [min(y.min(), y_pred.min()), max(y.max(), y_pred.max())]
ax.plot(lims, lims, 'r--', label='y = x')
ax.set_xlabel('Actual')
ax.set_ylabel('Predicted')
ax.set_title('Actual vs Predicted')
ax.legend()

ax = axes[0,1]
ax.scatter(y_pred, residuals, alpha=0.6, edgecolor='k')
ax.axhline(0, color='r', linestyle='--')
ax.set_xlabel('Predicted')
ax.set_ylabel('Residual (Actual - Pred)')
ax.set_title('Residuals vs Predicted')

ax = axes[1,0]
ax.hist(residuals, bins=30, color='C2', edgecolor='k')
ax.set_xlabel('Residual')
ax.set_title('Residuals Histogram')

# Feature importances if available
ax = axes[1,1]
if hasattr(model, 'feature_importances_'):
    importances = model.feature_importances_
    fn = feature_names[:len(importances)]
    order = np.argsort(importances)[::-1][:20]  # top 20
    ax.barh([fn[i] for i in order[::-1]], importances[order[::-1]], color='C3')
    ax.set_title('Feature importances (top)')
else:
    ax.text(0.5, 0.5, 'feature_importances_ not available', ha='center', va='center')
    ax.set_axis_off()

plt.tight_layout()
plt.show()

Notes:
- If the automatic `target` selection is not the value you want, change the `target` variable in the appropriate cell to the desired column name.
- You can swap to a different saved model by changing the `preferred` model filename or placing another `model_*.joblib` in the repo root.
- For more advanced plots (PDP, SHAP), consider adding shap or sklearn.inspection partial dependence plots as follow-ups.

In [ ]:
# Compare all saved models and produce plots per model
models = {}
for p in sorted(glob.glob('model_*.joblib')):
    try:
        m = joblib.load(p)
        models[p] = m
    except Exception as e:
        print('Failed to load', p, e)

if not models:
    print('No models found for comparison')
else:
    n = len(models)
    fig, axes = plt.subplots(n, 2, figsize=(12, 4 * n))
    if n == 1:
        axes = np.array([axes])

    for i, (p, m) in enumerate(models.items()):
        try:
            cols = getattr(m, 'feature_names_in_', None)
            if cols is not None:
                Xm = X.reindex(columns=cols).fillna(0)
            else:
                Xm = X
        except Exception:
            Xm = X

        try:
            yp = m.predict(Xm)
        except Exception as e:
            print('Prediction failed for', p, e)
            continue

        res = y - yp
        ax1 = axes[i, 0]
        ax2 = axes[i, 1]
        ax1.scatter(y, yp, alpha=0.6, edgecolor='k')
        lims = [min(y.min(), np.min(yp)), max(y.max(), np.max(yp))]
        ax1.plot(lims, lims, 'r--')
        ax1.set_title(f'Actual vs Predicted — {os.path.basename(p)}')
        ax1.set_xlabel('Actual')
        ax1.set_ylabel('Predicted')

        ax2.hist(res, bins=30, edgecolor='k')
        ax2.set_title(f'Residuals — {os.path.basename(p)}')
        ax2.set_xlabel('Residual')

    plt.tight_layout()
    plt.show()

    # Print summary metrics for each model
    for p, m in models.items():
        try:
            cols = getattr(m, 'feature_names_in_', None)
            Xm = X.reindex(columns=cols).fillna(0) if cols is not None else X
            yp = m.predict(Xm)
            print(os.path.basename(p), 'MSE:', f'{mean_squared_error(y,yp):.4f}', 'MAE:', f'{mean_absolute_error(y,yp):.4f}', 'R2:', f'{r2_score(y,yp):.4f}')
        except Exception as e:
            print('Failed metrics for', p, e)


In [ ]:
# Feature importances for models that provide them
for p in sorted(glob.glob('model_*.joblib')):
    try:
        m = joblib.load(p)
    except Exception as e:
        print('Could not load', p, e)
        continue

    if hasattr(m, 'feature_importances_'):
        importances = m.feature_importances_
        fn = list(X.columns)[:len(importances)]
        order = np.argsort(importances)[::-1][:20]
        plt.figure(figsize=(8, max(4, len(order)*0.25)))
        plt.title(f'Feature importances — {os.path.basename(p)}')
        plt.barh([fn[i] for i in order[::-1]], importances[order[::-1]], color='C3')
        plt.xlabel('Importance')
        plt.tight_layout()
        plt.show()
    else:
        print(os.path.basename(p), 'does not have feature_importances_')
